## Get Candidate Model Outputs

In [ ]:
!pip install langid
!pip install -U bitsandbytes

In [ ]:
import pandas as pd
import torch
from transformers import BartForConditionalGeneration, BartTokenizerFast
from transformers import AutoModelForCausalLM, AutoModelForSeq2SeqLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from datasets import load_dataset
import langid
import pickle
import random

### Load dataset (XSUM)

**Extreme Summarization (XSum) dataset** - Abstractive single sentence summaries of news articles from BBC

In [ ]:
# use Parquet files instead of old script
xsum = load_dataset("EdinburghNLP/xsum", revision="refs/convert/parquet")

In [ ]:
xsum_test = xsum["test"]
xsum_val = xsum["validation"]

test_docs = xsum["test"]["document"][:200]
val_docs = xsum["validation"]["document"][:100]

Check if articles are all in plain english:

In [ ]:
def is_english(text):
  lang, score = langid.classify(text.replace("\n", " ")[:500])
  return lang == "en"

In [ ]:
en_val_docs = [doc for doc in val_docs if is_english(doc)]
en_test_docs = [doc for doc in test_docs if is_english(doc)]

# first 200 test docs and first 100 validation docs are in englisch
#print(len(en_val_docs) == 100)
#print(len(en_test_docs) == 200)

### Load Target Model - DistilBART-xsum-12-6 - small summarization model from Hugging Face that can be run on a CPU only

In [ ]:
device = "cuda"

In [ ]:
model_name = "sshleifer/distilbart-xsum-12-6"
model = BartForConditionalGeneration.from_pretrained(model_name).to(device)
tokenizer = BartTokenizerFast.from_pretrained(model_name, add_prefix_space=True)

Change to false for test set calculation:

In [ ]:
use_validation = True

**Create an answer dictionary - results: <br>
For every instance we want the target model summary, the gold summary and later two more model summaries and a random summary. We also add the id and input document.**

In [ ]:
# model should be in evaluation mode
model.eval()

results = []

# validation or test set check
docs = val_docs if use_validation else test_docs
xsum = xsum_val if use_validation else xsum_test

for i in range(len(docs)):
  doc = docs[i]
  gold = xsum["summary"][i]

  # 1. Tokenize a single document
  inputs = tokenizer(
      doc,
      max_length=1024,
      truncation=True,
      return_tensors="pt"
      ).to(device)

  # 2. Generate the output tokens (max_new_tokens = default)
  summary_ids = model.generate(
      inputs["input_ids"],
      )

  # 3. Decode back into a string
  d_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

  # 4. Create dictionary for every instance
  instance = {
      "id": i,
      "input": doc,
      "summaries": {
          "gold": gold,
          "distilbart": d_summary.strip()
          }
      }

  results.append(instance)

# Get outputs from other models

### Llama-3-8b-Instruct

Setting in Paes et al. MExGen paper: <br>
API calls, greedy decoding, max_new_tokens = 100 <br>
**Summarization prompt**: <br>
Summarize the following article in one sentence. Do not preface the
summary with anything.

Use ***Llama-3-8b-Instruct-bnb-4bit***, because it won't fit in colab otherwise:

In [ ]:
model_id_llama = "unsloth/llama-3-8b-Instruct-bnb-4bit"

model_llama = AutoModelForCausalLM.from_pretrained(
    model_id_llama,
    device_map="auto",
)

tokenizer_llama = AutoTokenizer.from_pretrained(model_id_llama, use_fast=True)

In [ ]:
pipe = pipeline(
    "text-generation",
    model=model_llama,
    tokenizer=tokenizer_llama,
    return_full_text=False # only want to return completion
)

In [ ]:
def generate(instruction, system_prompt, pipe=pipe):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": instruction},
    ]

    prompt = tokenizer_llama.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    generation_parameters = {
        "max_new_tokens": 100,
        "do_sample": False,
        "return_full_text": False,
    }

    output = pipe(prompt, **generation_parameters)
    return output[0]["generated_text"]

In [ ]:
system_prompt = "You are a helpful summarization assistant."

docs = val_docs if use_validation else test_docs

for i in range(len(docs)):
  instruction = (
      "Summarize the following article in one sentence."
      "Do not preface the summary with anything.\n"
      f"Article: {docs[i]}"
      )
  model_sum = generate(instruction, system_prompt)
  results[i]["summaries"]["llama"] = model_sum

In [ ]:
#results[0]["summaries"]

###FLAN-T5-XL

Use this model **instead of FLAN-UL2 20B** (too big, even in 4-bit, and no access possible). <br>
Setting in paper: <br>
greedy decoding, max_new_tokens = 100, no chat template or system prompt was necessary to prompt Flan-UL2 to summarize

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,       # double quantization
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
model_id_flan = "google/flan-t5-xl"
tokenizer_flan = AutoTokenizer.from_pretrained(model_id_flan, use_fast=True)
model_flan = AutoModelForSeq2SeqLM.from_pretrained(
    model_id_flan,
    device_map="auto",
    quantization_config=bnb_config,
)
model_flan.eval()

In [ ]:
docs = val_docs if use_validation else test_docs

for i in range(len(docs)):
  inputs = tokenizer_flan("Summarize: " + docs[i], return_tensors="pt", truncation=True, max_length=1024).to(model_flan.device)

  with torch.no_grad():
    outputs = model_flan.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False # greedy
        )
  pred = tokenizer_flan.decode(outputs[0], skip_special_tokens=True)

  results[i]["summaries"]["flan"] = pred

In [ ]:
#results[0]["summaries"]

## Add random (gold) summaries as 5th option

In [ ]:
docs = val_docs if use_validation else test_docs
xsum_docs_summaries = xsum_val["summary"] if use_validation else xsum_test["summary"]

random.seed(42)

for i in range(len(docs)):
  idx = random.randrange(len(docs), len(xsum_docs_summaries))
  gold_rand = xsum_docs_summaries[idx]
  results[i]["summaries"]["random"] = gold_rand

In [ ]:
#results[0]["summaries"]

Save results to file:

In [ ]:
if use_validation:
  with open("INSTANCES_100_VAL.pkl", "wb") as f:
        pickle.dump(results, f)
else:
  with open("INSTANCES_200_TEST.pkl", "wb") as f:
    pickle.dump(results, f)